##### Complaint Agent Stream

This notebook creates a scheduled job to process complaints through the complaint agent

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
from databricks.sdk import WorkspaceClient
import databricks.sdk.service.jobs as j
import os
import sys

sys.path.append('../utils')
from agent_app_client import complaint_agent_app_name

w = WorkspaceClient()

CATALOG = dbutils.widgets.get("CATALOG")
COMPLAINT_AGENT_APP_NAME = complaint_agent_app_name(CATALOG)

notebook_abs_path = os.path.abspath("../jobs/complaint_agent_stream")
notebook_dbx_path = notebook_abs_path.replace(
    os.environ.get("DATABRICKS_WORKSPACE_ROOT", "/Workspace"),
    "/Workspace"
)

job_name = f"Complaint Agent Stream ({CATALOG})"

# timeout_seconds=600 (10 min, matches cron) so a single hung complaint-agent
# app call cannot block the whole queue forever.
task_def = [
    j.Task(
        task_key="complaint_agent_stream",
        timeout_seconds=600,
        notebook_task=j.NotebookTask(
            notebook_path=notebook_dbx_path,
            base_parameters={
                "CATALOG": CATALOG,
                "COMPLAINT_AGENT_APP_NAME": COMPLAINT_AGENT_APP_NAME,
            },
        )
    )
]
schedule_def = j.CronSchedule(
    quartz_cron_expression="0 0/10 * * * ?",
    timezone_id="UTC",
    pause_status=j.PauseStatus.UNPAUSED,
)

# queue.enabled=False: drop cron triggers if a previous run is still active
# instead of stacking them up. For an availableNow catch-up stream, dropping
# is correct: the NEXT tick will pick up whatever rows the previous run didn't
# get to.
queue_def = j.QueueSettings(enabled=False)

existing = [jb for jb in w.jobs.list(name=job_name) if jb.settings.name == job_name]
if existing:
    job_id = existing[0].job_id
    w.jobs.reset(job_id=job_id, new_settings=j.JobSettings(
        name=job_name, tasks=task_def, schedule=schedule_def, queue=queue_def,
    ))
    print(f"Updated existing job_id={job_id}")
else:
    job = w.jobs.create(name=job_name, tasks=task_def, schedule=schedule_def, queue=queue_def)
    job_id = job.job_id
    import sys
    sys.path.append('../utils')
    from uc_state import add
    add(CATALOG, "jobs", job)
    print(f"Created job_id={job_id}")

w.jobs.run_now(job_id=job_id)
print(f"Started run of {job_name} against app {COMPLAINT_AGENT_APP_NAME}")
